In [ ]:
%load ../lib/RecursiveSet.jsh
%load Formula.jsh

The file `Formula.jsh` defines the interface `Literal` and the record classes `Var` and `Neg`: A propositional variable $p$ is represented as `new Var("p")`, while the negated variable $\neg p$ is represented as `new Neg("p")`.  A clause is a set of literals, i.e. a `RecursiveSet<Literal>`, and a set of clauses has the type `RecursiveSet<RecursiveSet<Literal>>`.

In [ ]:
@SafeVarargs
<T> RecursiveSet<T> set(T... elements) {
    return new RecursiveSet<>(elements);
}

The function `first(S)` returns an arbitrary element from the set `S` in a non-random way.

In [ ]:
<T> T first(RecursiveSet<T> S) {
    return S.iterator().next();
}

The function `arb(S)` returns an arbitrary element from the set `S` in a random way.

In [ ]:
<T> T arb(RecursiveSet<T> S) {
    return S.pickRandom();
}

# The <a href="https://en.wikipedia.org/wiki/DPLL_algorithm">Davis-Putnam Algorithm</a>

This notebook implements the algorithm of Davis and Putnam.  Further details about this algorithm are provided in the lecture notes.

The function `complement(l)` computes the complement of a literal `l`.
If $p$ is a propositional variable, we have the following: 
* $\texttt{complement}(p) = \neg p$,
* $\texttt{complement}(\neg p) = p$.

As we are working with clauses that result form transforming given formulas into *conjunctive normal form* and these clauses do not contain $\top$ or $\bot$, we don't have to bother with $\top$ or $\bot$ in this function.

In [ ]:
Literal complement(Literal l) {
    return switch (l) {
        case Var(var p) -> new Neg(p);
        case Neg(var p) -> new Var(p);
        default -> throw new IllegalArgumentException("unknown literal: " + l);
    };
}

The function `extractVariable(l)` extracts the variable from the literal `l`.
If $p$ is a propositional variable, we have the following: 
* $\texttt{extractVariable}(p) = p$,
* $\texttt{extractVariable}(\neg p) = p$.

In [ ]:
String extractVariable(Literal l) {
    return switch (l) {
        case Var(var p) -> p;
        case Neg(var p) -> p;
        default -> throw new IllegalArgumentException("unknown literal: " + l);
    };
}

Given a set of clauses `Clauses` and a literal `l`, the procedure `reduce(Clauses, l)` performs all unit cuts and all unit subsumptions on clauses of of the set `Clauses` that are possible using the unit clause $\{\mathtt{l}\}$.  The resulting set of clauses is returned.  Mathematically, the function `reduce` is defined as follows:
$$\texttt{reduce}(\texttt{Clauses},l)  := 
 \Bigl\{\, C \backslash \bigl\{\overline{\,l\,}\bigr\} \;|\; C \in \texttt{Clauses} \wedge \overline{\,l\,} \in C \,\Bigr\} 
       \,\cup\, \Bigl\{\, C \in \texttt{Clauses} \mid \overline{\,l\,} \not\in C \wedge l \not\in C \Bigr\} \cup \bigl\{\{l\}\bigr\}.
$$
This function should only be called if the unit clause $\{l\}$ is an element of the set `Clauses`.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> reduce(RecursiveSet<RecursiveSet<Literal>> Clauses, Literal l) {
    var lBar   = complement(l);
    var result = Clauses.filterMap(
        cl -> !cl.contains(l),
        cl -> cl.contains(lBar) ? cl.difference(set(lBar)) : cl
    );
    result.add(set(l));
    return result;
}

`Clauses` is a set of clauses.  The call `saturate(Clauses)` computes the set of those clauses that can be derived from `Clauses` via repeated applications of unit cuts or unit subsumptions.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> saturate(RecursiveSet<RecursiveSet<Literal>> Clauses) {
    var S = Clauses;
    RecursiveSet<RecursiveSet<Literal>> Used = set();
    while (true) {
        var Units = S.filter(C -> C.size() == 1 && !Used.contains(C));
        if (Units.isEmpty()) {
            break;
        }
        var unit = arb(Units);
        Used.add(unit);
        S = reduce(S, first(unit));
    }
    return S;
}

The function `solve(Clauses)` takes a set of clauses  as input.  The function tries to compute a variable assignment that satisfies all clauses in `Clauses`.  If this is successful, a set of unit clauses is returned.  This set of unit clauses does not contain  any complementary literals and therefore corresponds to a variable assignment satisfying all clauses.  If the set `Clauses` is unsatisfiable, then the set `{{}}` is returned instead.

The real work is done by the function `solveRecursive`.  This function takes two additional arguments:
* `Variables` is the set of all variables occurring in `Clauses`.
* `UsedVars`  is the set of those variables that have already been used in case distinctions.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> solveRecursive(RecursiveSet<RecursiveSet<Literal>> Clauses,
                                                   RecursiveSet<String> Variables,
                                                   RecursiveSet<String> UsedVars) {
    var S = saturate(Clauses);
    RecursiveSet<Literal> EmptyClause = set();
    if (S.contains(EmptyClause))     { return set(EmptyClause); }
    if (S.every(C -> C.size() == 1)) { return S;                }
    var p            = arb(Variables.difference(UsedVars));
    var nextUsedVars = UsedVars.union(set(p));
    var Result1      = solveRecursive(S.union(set(set(new Var(p)))),
                                      Variables,
                                      nextUsedVars);
    if (!Result1.contains(EmptyClause)) { return Result1; }
    var pBar = new Neg(p);
    return solveRecursive(S.union(set(set(pBar))),
                          Variables,
                          nextUsedVars);
}

In [ ]:
RecursiveSet<RecursiveSet<Literal>> solve(RecursiveSet<RecursiveSet<Literal>> Clauses) {
    var Variables = flatMap(Clauses, C -> C.map(l -> extractVariable(l)));
    return solveRecursive(Clauses, Variables, set());
}

The function `solutionToString(S, Simplified)` takes a set of clauses `S` and the result `Simplified` of calling `solve(S)`.  It converts `Simplified` into a string that shows the variable assignment as a mapping of the variables to Boolean values.  This is only used for pretty printing.  The function `literalToString` converts a unit clause into a string.

Note that we can't call this function `toString`: Every class inherits the method `toString()` from the class `Object` and the methods defined in a notebook are compiled as methods of some class.

In [ ]:
String literalToString(RecursiveSet<Literal> C) {
    return switch (first(C)) {
        case Var(var p) -> p + " ↦ true";
        case Neg(var p) -> p + " ↦ false";
        default -> throw new IllegalArgumentException("not a unit clause: " + C);
    };
}

String solutionToString(RecursiveSet<RecursiveSet<Literal>> S,
                        RecursiveSet<RecursiveSet<Literal>> Simplified) {
    RecursiveSet<Literal> EmptyClause = set();
    if (Simplified.contains(EmptyClause)) {
        return S + " is unsolvable";
    }
    return Simplified.stream()
                     .map(C -> literalToString(C))
                     .collect(Collectors.joining(", ", "{ ", " }"));
}

The function `clause` creates a clause from the given literals.  In order to write down clauses succinctly, we define the variables `p`, `q`, `r`, and `s`.  The function `neg` takes a propositional variable $v$ and returns the negative literal $\neg v$.

In [ ]:
RecursiveSet<Literal> clause(Literal... literals) {
    return new RecursiveSet<>(literals);
}

In [ ]:
Literal neg(Var v) {
    return new Neg(v.name());
}

In [ ]:
var p = new Var("p");
var q = new Var("q");
var r = new Var("r");
var s = new Var("s");

In [ ]:
var c1 = clause(r, p, s);
var c2 = clause(r, s);
var c3 = clause(p, q, s);
var c4 = clause(neg(p), neg(q));
var c5 = clause(neg(p), s, neg(r));
var c6 = clause(p, neg(q), r);
var c7 = clause(neg(r), neg(s), q);
var c8 = clause(neg(p), neg(s));
var c9 = clause(p, neg(r), neg(q));
var c0 = clause(neg(p), r, q, neg(s));

var S = set(c0, c1, c2, c3, c4, c5, c6, c7, c8, c9);
System.out.println(solutionToString(S, solve(S)));

In [ ]:
var c11 = clause(p, r, q, neg(s));
var S = set(c0, c1, c2, c3, c4, c5, c6, c7, c8, c9, c11);
System.out.println(solutionToString(S, solve(S)));

In [ ]:
var c1  = clause(r, p, s);
var c2  = clause(r, s);
var c3  = clause(q, p, s);
var c4  = clause(neg(p), neg(q));
var c5  = clause(neg(p), s, neg(r));
var c6  = clause(p, neg(q), r);
var c7  = clause(neg(r), neg(s), q);
var c8  = clause(p, q, r, s);
var c9  = clause(r, neg(s), q);
var c10 = clause(s, neg(r), neg(q));
var c11 = clause(s, neg(r));
var c12 = clause(r, neg(s));

var S = set(c1, c2, c3, c4, c5, c6, c7, c8, c9, c10, c11, c12);
System.out.println(solutionToString(S, solve(S)));

The functions defined in this notebook are also available in the file `06-Davis-Putnam.jsh`.  This file is used by later notebooks.